# 01 - Data Wrangling

Notebook ini menjalankan proses **pure data wrangling**.

Alur utama:

1. **Gathering / Load Data**
2. **Assessing Data** satu per satu
3. **Cleaning Data**
4. **Validasi Relasi**
5. **Save Processed Dataset**

Batasan penting:
- Notebook ini **tidak menghitung ulang `stress_score`**.
- Notebook ini **tidak membuat ulang `stress_level`**.
- Notebook ini **tidak regenerate** `weekly_summaries`, `recommendations`, atau `insights`.
- Output sistem hanya dibersihkan dan divalidasi.

Alasan:
`stress_score` dan `stress_level` diperlakukan sebagai output sistem yang sudah tersedia. Tahap wrangling hanya membersihkan data kotor dan menjaga relasi tetap valid.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Setup Folder Project

In [2]:
current_path = Path.cwd().resolve()

if (current_path / "data" / "raw").exists():
    PROJECT_ROOT = current_path
else:
    # asumsi posisi notebook ada di data_analysis/notebooks/
    PROJECT_ROOT = current_path.parents[1]

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Users\User\Downloads\student_stress_data_science_pure_wrangling_project\student_stress_data_science_pure_wrangling
RAW_DIR      : C:\Users\User\Downloads\student_stress_data_science_pure_wrangling_project\student_stress_data_science_pure_wrangling\data\raw
PROCESSED_DIR: C:\Users\User\Downloads\student_stress_data_science_pure_wrangling_project\student_stress_data_science_pure_wrangling\data\processed
REPORT_DIR   : C:\Users\User\Downloads\student_stress_data_science_pure_wrangling_project\student_stress_data_science_pure_wrangling\outputs\reports


# A. Gathering / Load Data

Tahap ini hanya membaca semua dataset dari folder `data/raw/`.

Belum ada cleaning di tahap ini.

## 3. Load Semua Dataset Raw

In [3]:
users = pd.read_csv(RAW_DIR / "users.csv")
authentications = pd.read_csv(RAW_DIR / "authentications.csv")
daily_activities = pd.read_csv(RAW_DIR / "daily_activities.csv")
stress_predictions = pd.read_csv(RAW_DIR / "stress_predictions.csv")
weekly_summaries = pd.read_csv(RAW_DIR / "weekly_summaries.csv")
recommendations = pd.read_csv(RAW_DIR / "recommendations.csv")
insights = pd.read_csv(RAW_DIR / "insights.csv")

datasets = {
    "users": users,
    "authentications": authentications,
    "daily_activities": daily_activities,
    "stress_predictions": stress_predictions,
    "weekly_summaries": weekly_summaries,
    "recommendations": recommendations,
    "insights": insights
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

users: 300 rows, 7 columns
authentications: 300 rows, 6 columns
daily_activities: 27405 rows, 18 columns
stress_predictions: 27000 rows, 7 columns
weekly_summaries: 3600 rows, 13 columns
recommendations: 27198 rows, 10 columns
insights: 29965 rows, 7 columns


In [4]:
overview_df = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_full_rows": int(df.duplicated().sum())
    }
    for name, df in datasets.items()
])

overview_df

,dataset,rows,columns,missing_cells,duplicate_full_rows
0,users,300,7,81,0
1,authentications,300,6,0,0
2,daily_activities,27405,18,3806,0
3,stress_predictions,27000,7,0,0
4,weekly_summaries,3600,13,0,0
5,recommendations,27198,10,27198,0
6,insights,29965,7,29965,0


## Insight:
Semua dataset berhasil dimuat dari folder `data/raw/`.

`raw` bukan berarti semua dataset pasti kotor. `raw` berarti dataset belum melewati tahap assessing, cleaning, dan validasi.

# B. Assessing Data

Tahap ini mengecek kualitas dataset satu per satu menggunakan:

- `.head()`
- `.info()`
- `.describe(include="all")`
- missing value
- duplicate
- validasi khusus sesuai fungsi tabel

## 4. Helper Function untuk Assessing dan Cleaning

In [5]:
def parse_number(value):
    # Mengubah nilai seperti '7 jam', '8h', atau '150 mg' menjadi angka.
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    match = re.search(r"-?\d+(\.\d+)?", text)

    if match:
        return float(match.group(0))

    return np.nan


def parse_date(value):
    # Mengubah beberapa format tanggal menjadi datetime.
    if pd.isna(value):
        return pd.NaT

    text = str(value).strip()

    if text == "":
        return pd.NaT

    date = pd.to_datetime(text, errors="coerce")

    if pd.isna(date):
        date = pd.to_datetime(text, errors="coerce", dayfirst=True)

    return date

## 5. Assessing `users`

In [6]:
users.head()

,id,fullname,email,password_hash,profile_image,created_at,updated_at
0,1,Wahyu Saputra,wahyu.saputra1@example.com,96ace8d36b7a2c1423aa3886b70101fab26c0cd22733bd...,https://cdn.example.com/profiles/1.png,2025-08-19 01:00:00,2025-09-06 01:00:00
1,2,Rani Saputra,rani.saputra2@campusmail.id,54b54df2925579db43e616310494e29e6667dd72daa5a9...,https://cdn.example.com/profiles/2.png,2025-03-18 06:00:00,2025-06-02 06:00:00
2,3,Nadia Pratama,nadia.pratama3@example.com,5d2aea724fe1e5dd899fcb45f541f6b10c61c65ff5f292...,NaN,2025-08-24 16:00:00,2025-11-10 16:00:00
3,4,Alya Kurniawan,alya.kurniawan4@example.com,93523b0761950a989173d6870b82065494d82b9159d837...,https://cdn.example.com/profiles/4.png,2025-03-17 19:00:00,2025-04-15 19:00:00
4,5,Oki Permata,oki.permata5@studentmail.ac.id,d84def102e47bcb985144f28ca28e18e36a73cfd84e3ce...,https://cdn.example.com/profiles/5.png,2025-12-19 03:00:00,2026-03-19 03:00:00


In [7]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             300 non-null    int64
 1   fullname       300 non-null    str  
 2   email          300 non-null    str  
 3   password_hash  300 non-null    str  
 4   profile_image  219 non-null    str  
 5   created_at     300 non-null    str  
 6   updated_at     300 non-null    str  
dtypes: int64(1), str(6)
memory usage: 16.5 KB


In [8]:
users.describe(include='all')

,id,fullname,email,password_hash,profile_image,created_at,updated_at
count,300.000000,300,300,300,219,300,300
unique,NaN,219,300,300,219,295,293
top,NaN,Iqbal Febriani,wahyu.saputra1@example.com,96ace8d36b7a2c1423aa3886b70101fab26c0cd22733bd...,https://cdn.example.com/profiles/1.png,2025-08-09 07:00:00,2025-09-06 01:00:00
freq,NaN,5,1,1,1,2,2
mean,150.500000,NaN,NaN,NaN,NaN,NaN,NaN
std,86.746758,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,75.750000,NaN,NaN,NaN,NaN,NaN,NaN
50%,150.500000,NaN,NaN,NaN,NaN,NaN,NaN
75%,225.250000,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
users_missing = users.isna().sum()
users_duplicate_id = users["id"].duplicated().sum()
users_duplicate_email = users["email"].astype(str).str.strip().str.lower().duplicated().sum()
users_email_whitespace = users["email"].astype(str).str.match(r"^\s|\s$").sum()

print("Missing value:")
print(users_missing)

print("\nDuplicate id:", users_duplicate_id)
print("Duplicate email:", users_duplicate_email)
print("Email with whitespace:", users_email_whitespace)

Missing value:
id                0
fullname          0
email             0
password_hash     0
profile_image    81
created_at        0
updated_at        0
dtype: int64

Duplicate id: 0
Duplicate email: 0
Email with whitespace: 12


## Insight:
Dataset `users` bukan sumber dirty data utama.

Masalah yang mungkin muncul hanya bersifat ringan:
- whitespace pada `fullname` atau `email`
- kapitalisasi tidak seragam
- `profile_image` kosong

`profile_image` kosong tidak dianggap masalah fatal karena kolom ini bersifat opsional.

## 6. Assessing `authentications`

In [10]:
authentications.head()

,id,user_id,token,device_info,expires_at,created_at
0,1,1,b39fd303780b19b1942343c420cc16b723bb87f34daa35...,Firefox Linux,2026-05-10 14:00:00,2026-04-10 14:00:00
1,2,2,a515f8084a5160a9f8bd374a7312fceec1d56eac66f4ae...,Chrome Windows,2026-05-03 19:00:00,2026-04-03 19:00:00
2,3,3,1b80be6956ff3fff1043e0cff51b2d035bccc87da4e71f...,Firefox Linux,2026-05-07 03:00:00,2026-04-07 03:00:00
3,4,4,551d61b4f088322a0ba36c1a1e4b0561b6987b1d9fb18e...,Safari iOS,2026-05-10 07:00:00,2026-04-10 07:00:00
4,5,5,6545701c73c9744147dbe1cf51208000ef781d8e653f7c...,Chrome Windows,2026-05-10 13:00:00,2026-04-10 13:00:00


In [11]:
authentications.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           300 non-null    int64
 1   user_id      300 non-null    int64
 2   token        300 non-null    str  
 3   device_info  300 non-null    str  
 4   expires_at   300 non-null    str  
 5   created_at   300 non-null    str  
dtypes: int64(2), str(4)
memory usage: 14.2 KB


In [12]:
authentications.describe(include='all')

,id,user_id,token,device_info,expires_at,created_at
count,300.000000,300.000000,300,300,300,300
unique,NaN,NaN,300,5,180,180
top,NaN,NaN,b39fd303780b19b1942343c420cc16b723bb87f34daa35...,Safari iOS,2026-05-10 09:00:00,2026-04-10 09:00:00
freq,NaN,NaN,1,68,5,5
mean,150.500000,150.500000,NaN,NaN,NaN,NaN
std,86.746758,86.746758,NaN,NaN,NaN,NaN
min,1.000000,1.000000,NaN,NaN,NaN,NaN
25%,75.750000,75.750000,NaN,NaN,NaN,NaN
50%,150.500000,150.500000,NaN,NaN,NaN,NaN
75%,225.250000,225.250000,NaN,NaN,NaN,NaN


In [13]:
print("Missing value:")
print(authentications.isna().sum())

print("\nDuplicate id:", authentications["id"].duplicated().sum())
print("Missing token:", authentications["token"].isna().sum())
print("Missing expires_at:", authentications["expires_at"].isna().sum())

Missing value:
id             0
user_id        0
token          0
device_info    0
expires_at     0
created_at     0
dtype: int64

Duplicate id: 0
Missing token: 0
Missing expires_at: 0


## Insight:
Dataset `authentications` adalah output sistem, bukan input user harian.

Fokus assessing:
- validasi `user_id`
- validasi `token`
- validasi `expires_at`
- validasi timestamp

Dataset ini cukup dibersihkan ringan dan divalidasi relasinya ke `users_clean`.

## 7. Assessing `daily_activities`

In [14]:
daily_activities.head()

,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at
0,1,1,2026-01-01,7.23,1.93,5.78,3.81,13.0,160,4,7,5,4,6,2,5,2026-01-01 22:45:00,2026-01-01 23:39:00
1,2,1,2026-01-02,NaN,2.70,7.88,3.78,21.0,105,6,5,9,6,4,5,4,2026-01-02 19:44:00,2026-01-02 20:40:00
2,3,1,2026-01-03,6.56,3.24,8.65,4.78,23.0,123,6,6,5,5,8,2,6,2026-01-03 20:28:00,2026-01-03 20:33:00
3,4,1,2026-01-04,7.17,2.37,8.52,5.34,36.0,91,3,7,6,7,7,3,5,2026-01-04 23:55:00,2026-01-05 00:40:00
4,5,1,2026-01-05,7.6,5.20,7.43,4.82,29.0,103,6,6,7,9,2,5,6,2026-01-05 22:27:00,2026-01-05 23:02:00


In [15]:
daily_activities.info()

<class 'pandas.DataFrame'>
RangeIndex: 27405 entries, 0 to 27404
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         27405 non-null  int64  
 1   user_id                    27405 non-null  int64  
 2   activity_date              27405 non-null  str    
 3   sleep_hours                26857 non-null  str    
 4   study_hours                27405 non-null  float64
 5   screen_time_hours          27405 non-null  str    
 6   social_media_hours         26329 non-null  float64
 7   physical_activity_minutes  26591 non-null  float64
 8   caffeine_intake_mg         26037 non-null  str    
 9   mood_score                 27405 non-null  int64  
 10  fatigue_level              27405 non-null  int64  
 11  assignment_load            27405 non-null  int64  
 12  deadline_pressure          27405 non-null  int64  
 13  social_interaction_score   27405 non-null  int64  
 14  f

In [16]:
daily_activities.describe(include='all')

,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at
count,27405.0000,27405.000000,27405,26857,27405.000000,27405,26329.000000,26591.000000,26037,27405.000000,27405.000000,27405.000000,27405.000000,27405.000000,27405.000000,27405.000000,27405,27405
unique,NaN,NaN,339,988,NaN,1435,NaN,NaN,795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18418,19179
top,NaN,NaN,2026-03-06,28,NaN,7.18,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28 23:01:00,2026-01-01 23:55:00
freq,NaN,NaN,304,135,NaN,82,NaN,NaN,1033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,8
mean,13703.0000,150.518883,NaN,NaN,4.110956,NaN,3.063609,27.723440,NaN,4.392848,6.100383,6.640029,6.392885,6.397519,5.182339,6.093341,NaN,NaN
std,7911.2864,86.539895,NaN,NaN,1.656462,NaN,2.119260,17.565747,NaN,1.469842,1.518639,1.774117,1.900119,1.729177,1.875600,1.216902,NaN,NaN
min,1.0000,1.000000,NaN,NaN,0.000000,NaN,0.000000,-5.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN
25%,6852.0000,76.000000,NaN,NaN,2.960000,NaN,2.160000,14.000000,NaN,3.000000,5.000000,5.000000,5.000000,5.000000,4.000000,5.000000,NaN,NaN
50%,13703.0000,150.000000,NaN,NaN,4.060000,NaN,2.920000,27.000000,NaN,4.000000,6.000000,7.000000,6.000000,6.000000,5.000000,6.000000,NaN,NaN
75%,20554.0000,225.000000,NaN,NaN,5.200000,NaN,3.680000,40.000000,NaN,5.000000,7.000000,8.000000,8.000000,8.000000,6.000000,7.000000,NaN,NaN


In [17]:
print("Missing value:")
print(daily_activities.isna().sum())

print("\nDuplicate full rows:", daily_activities.duplicated().sum())
print("Duplicate user_id + activity_date:", daily_activities.duplicated(["user_id", "activity_date"]).sum())

Missing value:
id                              0
user_id                         0
activity_date                   0
sleep_hours                   548
study_hours                     0
screen_time_hours               0
social_media_hours           1076
physical_activity_minutes     814
caffeine_intake_mg           1368
mood_score                      0
fatigue_level                   0
assignment_load                 0
deadline_pressure               0
social_interaction_score        0
financial_worry_score           0
health_condition_score          0
created_at                      0
updated_at                      0
dtype: int64

Duplicate full rows: 0
Duplicate user_id + activity_date: 405


In [18]:
columns_to_check = [
    "activity_date",
    "sleep_hours",
    "screen_time_hours",
    "caffeine_intake_mg"
]

for col in columns_to_check:
    print(f"\nSample unique values from {col}:")
    print(daily_activities[col].dropna().astype(str).unique()[:15])


Sample unique values from activity_date:
<StringArray>
['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04', '2026-01-05',
 '2026-01-06', '2026-01-07', '2026-01-08', '2026-01-09', '2026-01-10',
 '2026-01-11', '2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15']
Length: 15, dtype: str

Sample unique values from sleep_hours:
<StringArray>
['7.23', '6.56', '7.17',  '7.6', '6.88', '7.14', '6.55', '7.05',  '7.1',
 '6.38', '6.61', '6.91', '6.52', '7.19', '7.41']
Length: 15, dtype: str

Sample unique values from screen_time_hours:
<StringArray>
[ '5.78',  '7.88',  '8.65',  '8.52',  '7.43',  '5.52',   '6.8',  '7.91',
 '7.98h',  '8.17',  '7.98',  '6.62',  '7.99', '11.36', '8.59h']
Length: 15, dtype: str

Sample unique values from caffeine_intake_mg:
<StringArray>
['160', '105', '123',  '91', '103', '230', '157',  '87',  '86',  '62', '170',
  '73', '199', '207', '196']
Length: 15, dtype: str


## Insight:
Dataset `daily_activities` adalah pusat dirty data karena merepresentasikan input user.

Problem yang perlu dibersihkan:
- missing value pada fitur aktivitas
- format angka campur string seperti `7 jam`, `8h`, atau `150 mg`
- format tanggal tidak konsisten
- duplicate submit pada `user_id + activity_date`
- nilai di luar range
- inkonsistensi logis seperti `social_media_hours > screen_time_hours`

Dataset ini membutuhkan cleaning penuh.

## 8. Assessing `stress_predictions`

In [19]:
stress_predictions.head()

,id,user_id,activity_id,prediction_date,stress_score,stress_level,created_at
0,1,1,1,2026-01-01,49.60,Medium,2026-01-01 23:58:00
1,2,1,2,2026-01-02,49.54,Medium,2026-01-02 20:43:00
2,3,1,3,2026-01-03,50.63,Medium,2026-01-03 20:49:00
3,4,1,4,2026-01-04,59.36,Medium,2026-01-05 00:56:00
4,5,1,5,2026-01-05,56.98,Medium,2026-01-05 23:09:00


In [20]:
stress_predictions.info()

<class 'pandas.DataFrame'>
RangeIndex: 27000 entries, 0 to 26999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               27000 non-null  int64  
 1   user_id          27000 non-null  int64  
 2   activity_id      27000 non-null  int64  
 3   prediction_date  27000 non-null  str    
 4   stress_score     27000 non-null  float64
 5   stress_level     27000 non-null  str    
 6   created_at       27000 non-null  str    
dtypes: float64(1), int64(3), str(3)
memory usage: 1.4 MB


In [21]:
stress_predictions.describe(include='all')

,id,user_id,activity_id,prediction_date,stress_score,stress_level,created_at
count,27000.00000,27000.000000,27000.00000,27000,27000.000000,27000,27000
unique,NaN,NaN,NaN,90,NaN,6,18991
top,NaN,NaN,NaN,2026-01-01,NaN,Medium,2026-02-02 23:19:00
freq,NaN,NaN,NaN,300,NaN,23509,6
mean,13500.50000,150.500000,13500.50000,NaN,56.036900,NaN,NaN
std,7794.37297,86.603663,7794.37297,NaN,9.551188,NaN,NaN
min,1.00000,1.000000,1.00000,NaN,21.000000,NaN,NaN
25%,6750.75000,75.750000,6750.75000,NaN,49.350000,NaN,NaN
50%,13500.50000,150.500000,13500.50000,NaN,55.400000,NaN,NaN
75%,20250.25000,225.250000,20250.25000,NaN,62.240000,NaN,NaN


In [22]:
stress_score_numeric = pd.to_numeric(stress_predictions["stress_score"], errors="coerce")
stress_level_standard = stress_predictions["stress_level"].astype(str).str.strip().str.title()

print("Missing value:")
print(stress_predictions.isna().sum())

print("\nDuplicate activity_id:", stress_predictions["activity_id"].duplicated().sum())
print("Stress level unique:", stress_predictions["stress_level"].unique())
print("Invalid stress_level count:", (~stress_level_standard.isin(["Low", "Medium", "High"])).sum())

print("\nStress score min:", stress_score_numeric.min())
print("Stress score max:", stress_score_numeric.max())

Missing value:
id                 0
user_id            0
activity_id        0
prediction_date    0
stress_score       0
stress_level       0
created_at         0
dtype: int64

Duplicate activity_id: 0
Stress level unique: <StringArray>
['Medium', 'High', 'Low', 'MEDIUM', 'LOW', 'HIGH']
Length: 6, dtype: str
Invalid stress_level count: 0

Stress score min: 21.0
Stress score max: 92.83


## Insight:
Dataset `stress_predictions` diperlakukan sebagai output sistem yang sudah tersedia.

Pada notebook wrangling ini:
- `stress_score` tidak dihitung ulang
- `stress_level` tidak dibuat ulang
- data hanya distandardisasi dan divalidasi

Karena `daily_activities` akan dibersihkan, beberapa `activity_id` pada `stress_predictions` bisa saja tidak lagi valid. Row seperti itu akan dibuang pada tahap cleaning untuk menjaga relasi.

## 9. Assessing `weekly_summaries`

In [23]:
weekly_summaries.head()

,id,user_id,week_start,week_end,average_stress_score,average_sleep_hours,average_screen_time,average_study_hours,high_stress_days,dominant_stress_level,stress_trend,main_trigger,created_at
0,1,1,2026-01-01,2026-01-07,52.92,7.10,7.23,3.46,0,Medium,Stable,Low Mood,2026-01-07 21:00:00
1,2,1,2026-01-08,2026-01-14,49.64,6.73,7.90,3.24,0,Medium,Stable,Academic Pressure,2026-01-14 21:00:00
2,3,1,2026-01-15,2026-01-21,50.77,7.53,8.34,3.56,0,Medium,Stable,Academic Pressure,2026-01-21 21:00:00
3,4,1,2026-01-22,2026-01-28,56.57,7.11,7.85,4.00,0,Medium,Decreasing,Low Mood,2026-01-28 21:00:00
4,5,1,2026-01-29,2026-02-04,67.72,6.63,8.76,4.89,3,Medium,Stable,Academic Pressure,2026-02-04 21:00:00


In [24]:
weekly_summaries.info()

<class 'pandas.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     3600 non-null   int64  
 1   user_id                3600 non-null   int64  
 2   week_start             3600 non-null   str    
 3   week_end               3600 non-null   str    
 4   average_stress_score   3600 non-null   float64
 5   average_sleep_hours    3600 non-null   float64
 6   average_screen_time    3600 non-null   float64
 7   average_study_hours    3600 non-null   float64
 8   high_stress_days       3600 non-null   int64  
 9   dominant_stress_level  3600 non-null   str    
 10  stress_trend           3600 non-null   str    
 11  main_trigger           3600 non-null   str    
 12  created_at             3600 non-null   str    
dtypes: float64(4), int64(3), str(6)
memory usage: 365.8 KB


In [25]:
weekly_summaries.describe(include='all')

,id,user_id,week_start,week_end,average_stress_score,average_sleep_hours,average_screen_time,average_study_hours,high_stress_days,dominant_stress_level,stress_trend,main_trigger,created_at
count,3600.000000,3600.00000,3600,3600,3600.000000,3600.000000,3600.000000,3600.000000,3600.000000,3600,3600,3600,3600
unique,NaN,NaN,12,12,NaN,NaN,NaN,NaN,NaN,3,6,10,12
top,NaN,NaN,2026-01-01,2026-01-07,NaN,NaN,NaN,NaN,NaN,Medium,Stable,Academic Pressure,2026-01-07 21:00:00
freq,NaN,NaN,300,300,NaN,NaN,NaN,NaN,NaN,3379,2144,1560,300
mean,1800.500000,150.50000,NaN,NaN,56.214336,6.734550,6.728019,4.138792,0.715000,NaN,NaN,NaN,NaN
std,1039.374812,86.61409,NaN,NaN,6.768336,0.790324,1.440070,1.310616,1.318809,NaN,NaN,NaN,NaN
min,1.000000,1.00000,NaN,NaN,40.490000,3.380000,2.450000,0.500000,0.000000,NaN,NaN,NaN,NaN
25%,900.750000,75.75000,NaN,NaN,51.470000,6.220000,5.820000,3.190000,0.000000,NaN,NaN,NaN,NaN
50%,1800.500000,150.50000,NaN,NaN,54.995000,6.730000,6.700000,4.050000,0.000000,NaN,NaN,NaN,NaN
75%,2700.250000,225.25000,NaN,NaN,59.862500,7.250000,7.740000,5.060000,1.000000,NaN,NaN,NaN,NaN


In [26]:
print("Missing value:")
print(weekly_summaries.isna().sum())

print("\nDuplicate user_id + week_start + week_end:")
print(weekly_summaries.duplicated(["user_id", "week_start", "week_end"]).sum())

print("\nStress trend unique:")
print(weekly_summaries["stress_trend"].unique())

print("\nDominant stress level unique:")
print(weekly_summaries["dominant_stress_level"].unique())

Missing value:
id                       0
user_id                  0
week_start               0
week_end                 0
average_stress_score     0
average_sleep_hours      0
average_screen_time      0
average_study_hours      0
high_stress_days         0
dominant_stress_level    0
stress_trend             0
main_trigger             0
created_at               0
dtype: int64

Duplicate user_id + week_start + week_end:
0

Stress trend unique:
<StringArray>
['Stable', 'Decreasing', 'Increasing', 'increasing', 'decreasing', 'stable']
Length: 6, dtype: str

Dominant stress level unique:
<StringArray>
['Medium', 'High', 'Low']
Length: 3, dtype: str


## Insight:
Dataset `weekly_summaries` adalah output agregasi mingguan.

Notebook ini tidak menghitung ulang weekly summary. Tahap wrangling hanya:
- standardisasi tipe data
- validasi user
- validasi periode mingguan
- validasi nilai kategorikal
- menghapus row yang tidak valid jika diperlukan

## 10. Assessing `recommendations`

In [27]:
recommendations.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
0,1,1,1.0,NaN,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-02 00:03:00
1,2,1,2.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-02 20:48:00
2,3,1,3.0,NaN,daily,digital_habit,Batasi screen time,Screen time lo tinggi. Kurangi penggunaan laya...,Medium,2026-01-03 20:54:00
3,4,1,4.0,NaN,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-05 01:01:00
4,5,1,5.0,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-05 23:14:00


In [28]:
recommendations.info()

<class 'pandas.DataFrame'>
RangeIndex: 27198 entries, 0 to 27197
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    27198 non-null  int64  
 1   user_id               27198 non-null  int64  
 2   stress_prediction_id  26365 non-null  float64
 3   weekly_summary_id     833 non-null    float64
 4   period_type           27198 non-null  str    
 5   category              27198 non-null  str    
 6   title                 27198 non-null  str    
 7   recommendation_text   27198 non-null  str    
 8   priority_level        27198 non-null  str    
 9   created_at            27198 non-null  str    
dtypes: float64(2), int64(2), str(6)
memory usage: 2.1 MB


In [29]:
recommendations.describe(include='all')

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
count,27198.000000,27198.000000,26365.000000,833.000000,27198,27198,27198,27198,27198,27198
unique,NaN,NaN,NaN,NaN,4,11,11,11,3,18669
top,NaN,NaN,NaN,NaN,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-03-18 21:05:00
freq,NaN,NaN,NaN,NaN,26287,12197,12197,12197,15086,158
mean,13599.500000,150.505147,13505.331766,1782.460984,NaN,NaN,NaN,NaN,NaN,NaN
std,7851.530647,86.662853,7796.995406,1051.702572,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,10.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,6800.250000,75.000000,6754.000000,827.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,13599.500000,151.000000,13528.000000,1834.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,20398.750000,226.000000,20258.000000,2685.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
print("Missing value:")
print(recommendations.isna().sum())

period_type = recommendations["period_type"].astype(str).str.strip().str.lower()

daily_rows = period_type == "daily"
weekly_rows = period_type == "weekly"

daily_source_valid = (
    recommendations.loc[daily_rows, "stress_prediction_id"].notna()
    & recommendations.loc[daily_rows, "weekly_summary_id"].isna()
)

weekly_source_valid = (
    recommendations.loc[weekly_rows, "weekly_summary_id"].notna()
    & recommendations.loc[weekly_rows, "stress_prediction_id"].isna()
)

print("\nPeriod type unique:")
print(recommendations["period_type"].unique())

print("\nDaily source valid:", daily_source_valid.all())
print("Weekly source valid:", weekly_source_valid.all())
print("Invalid period_type count:", (~period_type.isin(["daily", "weekly"])).sum())

Missing value:
id                          0
user_id                     0
stress_prediction_id      833
weekly_summary_id       26365
period_type                 0
category                    0
title                       0
recommendation_text         0
priority_level              0
created_at                  0
dtype: int64

Period type unique:
<StringArray>
['daily', 'Daily', 'weekly', 'Weekly']
Length: 4, dtype: str

Daily source valid: True
Weekly source valid: True
Invalid period_type count: 0


## Insight:
Missing value pada `recommendations` tidak otomatis berarti data kotor.

Aturan desain:
- Jika `period_type = daily`, maka `stress_prediction_id` terisi dan `weekly_summary_id` kosong.
- Jika `period_type = weekly`, maka `weekly_summary_id` terisi dan `stress_prediction_id` kosong.

Jadi missing pada salah satu source id adalah expected missing selama sesuai dengan `period_type`.

## 11. Assessing `insights`

In [31]:
insights.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at
0,1,1,1.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 00:01:00
1,2,1,2.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 20:46:00
2,3,1,3.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-03 20:52:00
3,4,1,4.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 00:59:00
4,5,1,5.0,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 23:12:00


In [32]:
insights.info()

<class 'pandas.DataFrame'>
RangeIndex: 29965 entries, 0 to 29964
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    29965 non-null  int64  
 1   user_id               29965 non-null  int64  
 2   stress_prediction_id  26365 non-null  float64
 3   weekly_summary_id     3600 non-null   float64
 4   period_type           29965 non-null  str    
 5   insight_text          29965 non-null  str    
 6   created_at            29965 non-null  str    
dtypes: float64(2), int64(2), str(3)
memory usage: 1.6 MB


In [33]:
insights.describe(include='all')

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at
count,29965.000000,29965.000000,26365.000000,3600.000000,29965,29965,29965
unique,NaN,NaN,NaN,NaN,4,78,18669
top,NaN,NaN,NaN,NaN,daily,Stress score hari ini sedang. Faktor yang perl...,2026-02-25 21:03:00
freq,NaN,NaN,NaN,NaN,26286,4531,303
mean,14983.000000,150.546805,13505.331766,1800.500000,NaN,NaN,NaN
std,8650.294744,86.629058,7796.995406,1039.374812,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN
25%,7492.000000,76.000000,6754.000000,900.750000,NaN,NaN,NaN
50%,14983.000000,151.000000,13528.000000,1800.500000,NaN,NaN,NaN
75%,22474.000000,226.000000,20258.000000,2700.250000,NaN,NaN,NaN


In [34]:
print("Missing value:")
print(insights.isna().sum())

period_type = insights["period_type"].astype(str).str.strip().str.lower()

daily_rows = period_type == "daily"
weekly_rows = period_type == "weekly"

daily_source_valid = (
    insights.loc[daily_rows, "stress_prediction_id"].notna()
    & insights.loc[daily_rows, "weekly_summary_id"].isna()
)

weekly_source_valid = (
    insights.loc[weekly_rows, "weekly_summary_id"].notna()
    & insights.loc[weekly_rows, "stress_prediction_id"].isna()
)

print("\nPeriod type unique:")
print(insights["period_type"].unique())

print("\nDaily source valid:", daily_source_valid.all())
print("Weekly source valid:", weekly_source_valid.all())
print("Invalid period_type count:", (~period_type.isin(["daily", "weekly"])).sum())

Missing value:
id                          0
user_id                     0
stress_prediction_id     3600
weekly_summary_id       26365
period_type                 0
insight_text                0
created_at                  0
dtype: int64

Period type unique:
<StringArray>
['daily', 'DAILY', 'weekly', 'WEEKLY']
Length: 4, dtype: str

Daily source valid: True
Weekly source valid: True
Invalid period_type count: 0


## Insight:
Sama seperti `recommendations`, missing pada salah satu source id di `insights` adalah expected missing selama sesuai dengan `period_type`.

Dataset ini tidak perlu generate ulang. Yang perlu dilakukan hanya standardisasi dan validasi relasi.

## 12. Problem Summary Setelah Assessing

In [35]:
problem_summary = pd.DataFrame([
    {
        "dataset": "users",
        "status": "problem ringan",
        "problem_found": users_email_whitespace > 0 or users_duplicate_id > 0 or users_duplicate_email > 0,
        "main_issue": "Ada kemungkinan whitespace atau format teks tidak rapi.",
        "action": "Trim whitespace, standardisasi email, validasi id dan email."
    },
    {
        "dataset": "authentications",
        "status": "validasi saja",
        "problem_found": False,
        "main_issue": "Tidak ada dirty data utama.",
        "action": "Validasi relasi user_id, token, expires_at, dan created_at."
    },
    {
        "dataset": "daily_activities",
        "status": "utama dirty data",
        "problem_found": True,
        "main_issue": "Missing value, format angka/tanggal tidak konsisten, duplicate submit, out of range, dan logical inconsistency.",
        "action": "Cleaning penuh: parsing, imputasi, clamp range, fix logic, dan deduplikasi."
    },
    {
        "dataset": "stress_predictions",
        "status": "output sistem / validasi relasi",
        "problem_found": False,
        "main_issue": "Bukan dirty data utama. Tidak dihitung ulang.",
        "action": "Standardisasi tipe data, validasi stress_score, stress_level, dan activity_id."
    },
    {
        "dataset": "weekly_summaries",
        "status": "output agregasi / validasi relasi",
        "problem_found": False,
        "main_issue": "Bukan dirty data utama. Tidak dihitung ulang.",
        "action": "Standardisasi tipe data, validasi user_id, periode, trend, dan duplicate weekly key."
    },
    {
        "dataset": "recommendations",
        "status": "validasi source id",
        "problem_found": False,
        "main_issue": "Expected missing pada source id sesuai period_type.",
        "action": "Validasi daily/weekly source dan period_type."
    },
    {
        "dataset": "insights",
        "status": "validasi source id",
        "problem_found": False,
        "main_issue": "Expected missing pada source id sesuai period_type.",
        "action": "Validasi daily/weekly source dan period_type."
    },
])

problem_summary

,dataset,status,problem_found,main_issue,action
0,users,problem ringan,True,Ada kemungkinan whitespace atau format teks ti...,"Trim whitespace, standardisasi email, validasi..."
1,authentications,validasi saja,False,Tidak ada dirty data utama.,"Validasi relasi user_id, token, expires_at, da..."
2,daily_activities,utama dirty data,True,"Missing value, format angka/tanggal tidak kons...","Cleaning penuh: parsing, imputasi, clamp range..."
3,stress_predictions,output sistem / validasi relasi,False,Bukan dirty data utama. Tidak dihitung ulang.,"Standardisasi tipe data, validasi stress_score..."
4,weekly_summaries,output agregasi / validasi relasi,False,Bukan dirty data utama. Tidak dihitung ulang.,"Standardisasi tipe data, validasi user_id, per..."
5,recommendations,validasi source id,False,Expected missing pada source id sesuai period_...,Validasi daily/weekly source dan period_type.
6,insights,validasi source id,False,Expected missing pada source id sesuai period_...,Validasi daily/weekly source dan period_type.


## Insight:
Masalah data utama hanya ada pada `daily_activities`.

Dataset lain tetap diproses, tetapi statusnya lebih tepat disebut:
- validasi ringan
- standardisasi format
- validasi relasi
- validasi expected missing

Tidak ada kalkulasi ulang `stress_score` pada notebook ini.

# C. Cleaning Data

Tahap cleaning dilakukan berdasarkan hasil assessing.

Prinsip:
- `daily_activities.id` dipertahankan dari raw.
- Jika duplicate `user_id + activity_date`, dipilih record terbaru berdasarkan `updated_at`.
- Output sistem tidak dihitung ulang.
- Row output sistem yang relasinya tidak valid akan dibuang.

## 13. Cleaning `users`

In [36]:
users_clean = users.copy()

# Membersihkan teks
users_clean["fullname"] = users_clean["fullname"].astype(str).str.strip().str.title()
users_clean["email"] = users_clean["email"].astype(str).str.strip().str.lower()
users_clean["password_hash"] = users_clean["password_hash"].astype(str).str.strip()
users_clean["profile_image"] = users_clean["profile_image"].fillna("").astype(str).str.strip()

# Parse tipe data
users_clean["id"] = pd.to_numeric(users_clean["id"], errors="coerce")
users_clean["created_at"] = pd.to_datetime(users_clean["created_at"], errors="coerce")
users_clean["updated_at"] = pd.to_datetime(users_clean["updated_at"], errors="coerce")

# Hapus data fatal
users_clean = users_clean.dropna(subset=["id", "email", "password_hash"])

# Hapus duplicate
users_clean = users_clean.drop_duplicates(subset=["id"], keep="last")
users_clean = users_clean.drop_duplicates(subset=["email"], keep="last")

# Format final
users_clean["id"] = users_clean["id"].astype(int)
users_clean["created_at"] = users_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
users_clean["updated_at"] = users_clean["updated_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

users_clean = users_clean[
    ["id", "fullname", "email", "password_hash", "profile_image", "created_at", "updated_at"]
].sort_values("id")

users_clean.head()

,id,fullname,email,password_hash,profile_image,created_at,updated_at
0,1,Wahyu Saputra,wahyu.saputra1@example.com,96ace8d36b7a2c1423aa3886b70101fab26c0cd22733bd...,https://cdn.example.com/profiles/1.png,2025-08-19 01:00:00,2025-09-06 01:00:00
1,2,Rani Saputra,rani.saputra2@campusmail.id,54b54df2925579db43e616310494e29e6667dd72daa5a9...,https://cdn.example.com/profiles/2.png,2025-03-18 06:00:00,2025-06-02 06:00:00
2,3,Nadia Pratama,nadia.pratama3@example.com,5d2aea724fe1e5dd899fcb45f541f6b10c61c65ff5f292...,,2025-08-24 16:00:00,2025-11-10 16:00:00
3,4,Alya Kurniawan,alya.kurniawan4@example.com,93523b0761950a989173d6870b82065494d82b9159d837...,https://cdn.example.com/profiles/4.png,2025-03-17 19:00:00,2025-04-15 19:00:00
4,5,Oki Permata,oki.permata5@studentmail.ac.id,d84def102e47bcb985144f28ca28e18e36a73cfd84e3ce...,https://cdn.example.com/profiles/5.png,2025-12-19 03:00:00,2026-03-19 03:00:00


## Insight:
`users_clean` hanya mengalami standardisasi ringan. Tidak ada transformasi agresif karena tabel ini bukan sumber dirty data utama.

## 14. Cleaning `authentications`

In [37]:
authentications_clean = authentications.copy()

authentications_clean["id"] = pd.to_numeric(authentications_clean["id"], errors="coerce")
authentications_clean["user_id"] = pd.to_numeric(authentications_clean["user_id"], errors="coerce")
authentications_clean["token"] = authentications_clean["token"].astype(str).str.strip()
authentications_clean["device_info"] = authentications_clean["device_info"].fillna("").astype(str).str.strip()
authentications_clean["expires_at"] = pd.to_datetime(authentications_clean["expires_at"], errors="coerce")
authentications_clean["created_at"] = pd.to_datetime(authentications_clean["created_at"], errors="coerce")

# Hapus data fatal
authentications_clean = authentications_clean.dropna(subset=["id", "user_id", "token", "expires_at"])

# Validasi user_id terhadap users_clean
valid_user_ids = set(users_clean["id"])
authentications_clean = authentications_clean[
    authentications_clean["user_id"].isin(valid_user_ids)
]

# Hapus duplicate id jika ada
authentications_clean = authentications_clean.drop_duplicates(subset=["id"], keep="last")

# Format final
authentications_clean["id"] = authentications_clean["id"].astype(int)
authentications_clean["user_id"] = authentications_clean["user_id"].astype(int)
authentications_clean["expires_at"] = authentications_clean["expires_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
authentications_clean["created_at"] = authentications_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

authentications_clean = authentications_clean[
    ["id", "user_id", "token", "device_info", "expires_at", "created_at"]
].sort_values("id")

authentications_clean.head()

,id,user_id,token,device_info,expires_at,created_at
0,1,1,b39fd303780b19b1942343c420cc16b723bb87f34daa35...,Firefox Linux,2026-05-10 14:00:00,2026-04-10 14:00:00
1,2,2,a515f8084a5160a9f8bd374a7312fceec1d56eac66f4ae...,Chrome Windows,2026-05-03 19:00:00,2026-04-03 19:00:00
2,3,3,1b80be6956ff3fff1043e0cff51b2d035bccc87da4e71f...,Firefox Linux,2026-05-07 03:00:00,2026-04-07 03:00:00
3,4,4,551d61b4f088322a0ba36c1a1e4b0561b6987b1d9fb18e...,Safari iOS,2026-05-10 07:00:00,2026-04-10 07:00:00
4,5,5,6545701c73c9744147dbe1cf51208000ef781d8e653f7c...,Chrome Windows,2026-05-10 13:00:00,2026-04-10 13:00:00


## Insight:
`authentications_clean` diproses dengan validasi relasi ke `users_clean`. Tidak ada data baru yang dibuat.

## 15. Cleaning `daily_activities`

In [38]:
daily_clean = daily_activities.copy()
raw_daily_rows = len(daily_clean)

# Parse id dan tanggal
daily_clean["id"] = pd.to_numeric(daily_clean["id"], errors="coerce")
daily_clean["user_id"] = pd.to_numeric(daily_clean["user_id"], errors="coerce")
daily_clean["activity_date"] = daily_clean["activity_date"].apply(parse_date)

# Kolom numerik
hour_columns = [
    "sleep_hours",
    "study_hours",
    "screen_time_hours",
    "social_media_hours"
]

score_columns = [
    "mood_score",
    "fatigue_level",
    "assignment_load",
    "deadline_pressure",
    "social_interaction_score",
    "financial_worry_score",
    "health_condition_score"
]

other_numeric_columns = [
    "physical_activity_minutes",
    "caffeine_intake_mg"
]

numeric_columns = hour_columns + score_columns + other_numeric_columns

# Parse angka dari format campuran
for col in numeric_columns:
    daily_clean[col] = daily_clean[col].apply(parse_number)

# Parse timestamp
daily_clean["created_at"] = pd.to_datetime(daily_clean["created_at"], errors="coerce")
daily_clean["updated_at"] = pd.to_datetime(daily_clean["updated_at"], errors="coerce")

# Hapus baris dengan key/tanggal rusak
daily_clean = daily_clean.dropna(subset=["id", "user_id", "activity_date"])

# Pastikan user_id valid
daily_clean = daily_clean[daily_clean["user_id"].isin(valid_user_ids)]

# Cek missing sebelum imputasi
missing_before_imputation = daily_clean[numeric_columns].isna().sum()
missing_before_imputation

C:\Users\User\AppData\Local\Temp\ipykernel_28520\2981232923.py:29: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  date = pd.to_datetime(text, errors="coerce")


sleep_hours                   548
study_hours                     0
screen_time_hours               0
social_media_hours           1076
mood_score                      0
fatigue_level                   0
assignment_load                 0
deadline_pressure               0
social_interaction_score        0
financial_worry_score           0
health_condition_score          0
physical_activity_minutes     814
caffeine_intake_mg           1368
dtype: int64

In [39]:
# Imputasi missing value
# Prioritas: median per user, lalu median global
for col in numeric_columns:
    median_per_user = daily_clean.groupby("user_id")[col].transform("median")
    global_median = daily_clean[col].median()

    daily_clean[col] = daily_clean[col].fillna(median_per_user)
    daily_clean[col] = daily_clean[col].fillna(global_median)

# Clamp kolom jam ke range 0-24
for col in hour_columns:
    daily_clean[col] = daily_clean[col].clip(0, 24)

# Clamp kolom skor ke range 1-10
for col in score_columns:
    daily_clean[col] = daily_clean[col].clip(1, 10)

# Nilai ini tidak boleh negatif
daily_clean["physical_activity_minutes"] = daily_clean["physical_activity_minutes"].clip(lower=0)
daily_clean["caffeine_intake_mg"] = daily_clean["caffeine_intake_mg"].clip(lower=0)

# Perbaiki inkonsistensi logika:
# social_media_hours tidak boleh lebih besar dari screen_time_hours
daily_clean["social_media_hours"] = np.minimum(
    daily_clean["social_media_hours"],
    daily_clean["screen_time_hours"]
)

# Deduplikasi user_id + activity_date
# Simpan record terbaru berdasarkan updated_at
daily_clean["sort_time"] = daily_clean["updated_at"].fillna(daily_clean["created_at"])
daily_clean = daily_clean.sort_values(["user_id", "activity_date", "sort_time"])

duplicates_removed = daily_clean.duplicated(
    subset=["user_id", "activity_date"],
    keep="last"
).sum()

daily_clean = daily_clean.drop_duplicates(
    subset=["user_id", "activity_date"],
    keep="last"
)

# Penting:
# id TIDAK di-reassign agar relasi ke stress_predictions.activity_id tetap bisa divalidasi.
daily_clean = daily_clean.sort_values(["user_id", "activity_date"]).reset_index(drop=True)

# Format final
daily_clean["id"] = daily_clean["id"].astype(int)
daily_clean["user_id"] = daily_clean["user_id"].astype(int)
daily_clean["activity_date"] = daily_clean["activity_date"].dt.strftime("%Y-%m-%d")

for col in hour_columns:
    daily_clean[col] = daily_clean[col].round(2)

for col in score_columns + other_numeric_columns:
    daily_clean[col] = daily_clean[col].round().astype(int)

daily_clean["created_at"] = daily_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
daily_clean["updated_at"] = daily_clean["updated_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

daily_clean = daily_clean[
    [
        "id", "user_id", "activity_date",
        "sleep_hours", "study_hours", "screen_time_hours", "social_media_hours",
        "physical_activity_minutes", "caffeine_intake_mg",
        "mood_score", "fatigue_level", "assignment_load", "deadline_pressure",
        "social_interaction_score", "financial_worry_score", "health_condition_score",
        "created_at", "updated_at"
    ]
]

print("Raw rows:", raw_daily_rows)
print("Clean rows:", len(daily_clean))
print("Duplicates removed:", duplicates_removed)

daily_clean.head()

Raw rows: 27405
Clean rows: 26984
Duplicates removed: 421


,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at
0,1,1,2026-01-01,7.23,1.93,5.78,3.81,13,160,4,7,5,4,6,2,5,2026-01-01 22:45:00,2026-01-01 23:39:00
1,2,1,2026-01-02,7.10,2.70,7.88,3.78,21,105,6,5,9,6,4,5,4,2026-01-02 19:44:00,2026-01-02 20:40:00
2,3,1,2026-01-03,6.56,3.24,8.65,4.78,23,123,6,6,5,5,8,2,6,2026-01-03 20:28:00,2026-01-03 20:33:00
3,4,1,2026-01-04,7.17,2.37,8.52,5.34,36,91,3,7,6,7,7,3,5,2026-01-04 23:55:00,2026-01-05 00:40:00
4,5,1,2026-01-05,7.60,5.20,7.43,4.82,29,103,6,6,7,9,2,5,6,2026-01-05 22:27:00,2026-01-05 23:02:00


## Insight:
`daily_activities_clean` adalah hasil cleaning utama.

Poin penting:
- ID asli dari raw tetap dipertahankan.
- Duplicate `user_id + activity_date` dibersihkan dengan memilih record terbaru.
- Tidak ada reassign ID, supaya relasi ke `stress_predictions.activity_id` tidak rusak.

## 16. Cleaning `stress_predictions` Tanpa Menghitung Ulang Score

In [40]:
stress_predictions_clean = stress_predictions.copy()

# Parse tipe data
stress_predictions_clean["id"] = pd.to_numeric(stress_predictions_clean["id"], errors="coerce")
stress_predictions_clean["user_id"] = pd.to_numeric(stress_predictions_clean["user_id"], errors="coerce")
stress_predictions_clean["activity_id"] = pd.to_numeric(stress_predictions_clean["activity_id"], errors="coerce")
stress_predictions_clean["prediction_date"] = pd.to_datetime(stress_predictions_clean["prediction_date"], errors="coerce")
stress_predictions_clean["stress_score"] = pd.to_numeric(stress_predictions_clean["stress_score"], errors="coerce")
stress_predictions_clean["stress_level"] = stress_predictions_clean["stress_level"].astype(str).str.strip().str.title()
stress_predictions_clean["created_at"] = pd.to_datetime(stress_predictions_clean["created_at"], errors="coerce")

# Hapus data fatal
stress_predictions_clean = stress_predictions_clean.dropna(
    subset=["id", "user_id", "activity_id", "prediction_date", "stress_score", "stress_level", "created_at"]
)

# Validasi stress_level
stress_predictions_clean = stress_predictions_clean[
    stress_predictions_clean["stress_level"].isin(["Low", "Medium", "High"])
]

# Clamp stress_score agar tetap 0-100
stress_predictions_clean["stress_score"] = stress_predictions_clean["stress_score"].clip(0, 100)

# Validasi relasi user_id dan activity_id
valid_activity_ids = set(daily_clean["id"])

stress_predictions_clean = stress_predictions_clean[
    stress_predictions_clean["user_id"].isin(valid_user_ids)
    & stress_predictions_clean["activity_id"].isin(valid_activity_ids)
]

# Pastikan activity_id unique
stress_predictions_clean = stress_predictions_clean.sort_values("created_at")
stress_predictions_clean = stress_predictions_clean.drop_duplicates(subset=["activity_id"], keep="last")

# Format final
stress_predictions_clean["id"] = stress_predictions_clean["id"].astype(int)
stress_predictions_clean["user_id"] = stress_predictions_clean["user_id"].astype(int)
stress_predictions_clean["activity_id"] = stress_predictions_clean["activity_id"].astype(int)
stress_predictions_clean["prediction_date"] = stress_predictions_clean["prediction_date"].dt.strftime("%Y-%m-%d")
stress_predictions_clean["created_at"] = stress_predictions_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

stress_predictions_clean = stress_predictions_clean[
    ["id", "user_id", "activity_id", "prediction_date", "stress_score", "stress_level", "created_at"]
].sort_values("id")

stress_predictions_clean.head()

,id,user_id,activity_id,prediction_date,stress_score,stress_level,created_at
0,1,1,1,2026-01-01,49.60,Medium,2026-01-01 23:58:00
1,2,1,2,2026-01-02,49.54,Medium,2026-01-02 20:43:00
2,3,1,3,2026-01-03,50.63,Medium,2026-01-03 20:49:00
3,4,1,4,2026-01-04,59.36,Medium,2026-01-05 00:56:00
4,5,1,5,2026-01-05,56.98,Medium,2026-01-05 23:09:00


## Insight:
`stress_predictions_clean` tidak dihitung ulang.

Yang dilakukan hanya:
- parse tipe data
- standardisasi `stress_level`
- clamp `stress_score` ke 0-100
- validasi `user_id`
- validasi `activity_id`
- hapus prediction yang tidak lagi punya daily activity setelah cleaning

Ini menjaga wrangling tetap murni sebagai proses cleaning dan validasi.

## 17. Cleaning `weekly_summaries` Tanpa Menghitung Ulang Agregasi

In [41]:
weekly_summaries_clean = weekly_summaries.copy()

# Parse tipe data
weekly_summaries_clean["id"] = pd.to_numeric(weekly_summaries_clean["id"], errors="coerce")
weekly_summaries_clean["user_id"] = pd.to_numeric(weekly_summaries_clean["user_id"], errors="coerce")
weekly_summaries_clean["week_start"] = pd.to_datetime(weekly_summaries_clean["week_start"], errors="coerce")
weekly_summaries_clean["week_end"] = pd.to_datetime(weekly_summaries_clean["week_end"], errors="coerce")
weekly_summaries_clean["average_stress_score"] = pd.to_numeric(weekly_summaries_clean["average_stress_score"], errors="coerce")
weekly_summaries_clean["average_sleep_hours"] = pd.to_numeric(weekly_summaries_clean["average_sleep_hours"], errors="coerce")
weekly_summaries_clean["average_screen_time"] = pd.to_numeric(weekly_summaries_clean["average_screen_time"], errors="coerce")
weekly_summaries_clean["average_study_hours"] = pd.to_numeric(weekly_summaries_clean["average_study_hours"], errors="coerce")
weekly_summaries_clean["high_stress_days"] = pd.to_numeric(weekly_summaries_clean["high_stress_days"], errors="coerce")
weekly_summaries_clean["dominant_stress_level"] = weekly_summaries_clean["dominant_stress_level"].astype(str).str.strip().str.title()
weekly_summaries_clean["stress_trend"] = weekly_summaries_clean["stress_trend"].astype(str).str.strip().str.title()
weekly_summaries_clean["main_trigger"] = weekly_summaries_clean["main_trigger"].astype(str).str.strip()
weekly_summaries_clean["created_at"] = pd.to_datetime(weekly_summaries_clean["created_at"], errors="coerce")

# Hapus data fatal
weekly_summaries_clean = weekly_summaries_clean.dropna(
    subset=[
        "id", "user_id", "week_start", "week_end",
        "average_stress_score", "high_stress_days",
        "dominant_stress_level", "stress_trend", "created_at"
    ]
)

# Validasi user_id
weekly_summaries_clean = weekly_summaries_clean[
    weekly_summaries_clean["user_id"].isin(valid_user_ids)
]

# Validasi kategori
weekly_summaries_clean = weekly_summaries_clean[
    weekly_summaries_clean["dominant_stress_level"].isin(["Low", "Medium", "High"])
]

weekly_summaries_clean = weekly_summaries_clean[
    weekly_summaries_clean["stress_trend"].isin(["Increasing", "Stable", "Decreasing"])
]

# Validasi range
weekly_summaries_clean["average_stress_score"] = weekly_summaries_clean["average_stress_score"].clip(0, 100)
weekly_summaries_clean["average_sleep_hours"] = weekly_summaries_clean["average_sleep_hours"].clip(0, 24)
weekly_summaries_clean["average_screen_time"] = weekly_summaries_clean["average_screen_time"].clip(0, 24)
weekly_summaries_clean["average_study_hours"] = weekly_summaries_clean["average_study_hours"].clip(0, 24)
weekly_summaries_clean["high_stress_days"] = weekly_summaries_clean["high_stress_days"].clip(0, 7)

# week_end harus >= week_start
weekly_summaries_clean = weekly_summaries_clean[
    weekly_summaries_clean["week_end"] >= weekly_summaries_clean["week_start"]
]

# Unique weekly key
weekly_summaries_clean = weekly_summaries_clean.sort_values("created_at")
weekly_summaries_clean = weekly_summaries_clean.drop_duplicates(
    subset=["user_id", "week_start", "week_end"],
    keep="last"
)

# Format final
weekly_summaries_clean["id"] = weekly_summaries_clean["id"].astype(int)
weekly_summaries_clean["user_id"] = weekly_summaries_clean["user_id"].astype(int)
weekly_summaries_clean["high_stress_days"] = weekly_summaries_clean["high_stress_days"].round().astype(int)
weekly_summaries_clean["week_start"] = weekly_summaries_clean["week_start"].dt.strftime("%Y-%m-%d")
weekly_summaries_clean["week_end"] = weekly_summaries_clean["week_end"].dt.strftime("%Y-%m-%d")
weekly_summaries_clean["created_at"] = weekly_summaries_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

weekly_summaries_clean = weekly_summaries_clean[
    [
        "id", "user_id", "week_start", "week_end",
        "average_stress_score", "average_sleep_hours", "average_screen_time",
        "average_study_hours", "high_stress_days", "dominant_stress_level",
        "stress_trend", "main_trigger", "created_at"
    ]
].sort_values("id")

weekly_summaries_clean.head()

,id,user_id,week_start,week_end,average_stress_score,average_sleep_hours,average_screen_time,average_study_hours,high_stress_days,dominant_stress_level,stress_trend,main_trigger,created_at
0,1,1,2026-01-01,2026-01-07,52.92,7.10,7.23,3.46,0,Medium,Stable,Low Mood,2026-01-07 21:00:00
1,2,1,2026-01-08,2026-01-14,49.64,6.73,7.90,3.24,0,Medium,Stable,Academic Pressure,2026-01-14 21:00:00
2,3,1,2026-01-15,2026-01-21,50.77,7.53,8.34,3.56,0,Medium,Stable,Academic Pressure,2026-01-21 21:00:00
3,4,1,2026-01-22,2026-01-28,56.57,7.11,7.85,4.00,0,Medium,Decreasing,Low Mood,2026-01-28 21:00:00
4,5,1,2026-01-29,2026-02-04,67.72,6.63,8.76,4.89,3,Medium,Stable,Academic Pressure,2026-02-04 21:00:00


## Insight:
`weekly_summaries_clean` tidak dihitung ulang.

Yang dilakukan hanya standardisasi dan validasi. Ini menjaga laporan tetap defensible karena wrangling tidak membuat agregasi baru.

## 18. Cleaning `recommendations`

In [42]:
recommendations_clean = recommendations.copy()

# Parse tipe data
recommendations_clean["id"] = pd.to_numeric(recommendations_clean["id"], errors="coerce")
recommendations_clean["user_id"] = pd.to_numeric(recommendations_clean["user_id"], errors="coerce")
recommendations_clean["stress_prediction_id"] = pd.to_numeric(recommendations_clean["stress_prediction_id"], errors="coerce")
recommendations_clean["weekly_summary_id"] = pd.to_numeric(recommendations_clean["weekly_summary_id"], errors="coerce")
recommendations_clean["period_type"] = recommendations_clean["period_type"].astype(str).str.strip().str.lower()
recommendations_clean["category"] = recommendations_clean["category"].astype(str).str.strip()
recommendations_clean["title"] = recommendations_clean["title"].astype(str).str.strip()
recommendations_clean["recommendation_text"] = recommendations_clean["recommendation_text"].astype(str).str.strip()
recommendations_clean["priority_level"] = recommendations_clean["priority_level"].astype(str).str.strip().str.title()
recommendations_clean["created_at"] = pd.to_datetime(recommendations_clean["created_at"], errors="coerce")

# Hapus data fatal umum
recommendations_clean = recommendations_clean.dropna(
    subset=["id", "user_id", "period_type", "category", "title", "recommendation_text", "priority_level", "created_at"]
)

# Validasi user dan period_type
recommendations_clean = recommendations_clean[
    recommendations_clean["user_id"].isin(valid_user_ids)
]

recommendations_clean = recommendations_clean[
    recommendations_clean["period_type"].isin(["daily", "weekly"])
]

# Validasi source id sesuai period_type
valid_prediction_ids = set(stress_predictions_clean["id"])
valid_weekly_ids = set(weekly_summaries_clean["id"])

daily_mask = recommendations_clean["period_type"] == "daily"
weekly_mask = recommendations_clean["period_type"] == "weekly"

daily_valid = (
    daily_mask
    & recommendations_clean["stress_prediction_id"].isin(valid_prediction_ids)
    & recommendations_clean["weekly_summary_id"].isna()
)

weekly_valid = (
    weekly_mask
    & recommendations_clean["weekly_summary_id"].isin(valid_weekly_ids)
    & recommendations_clean["stress_prediction_id"].isna()
)

recommendations_clean = recommendations_clean[daily_valid | weekly_valid]

# Format final
recommendations_clean["id"] = recommendations_clean["id"].astype(int)
recommendations_clean["user_id"] = recommendations_clean["user_id"].astype(int)

recommendations_clean["stress_prediction_id"] = recommendations_clean["stress_prediction_id"].astype("Int64")
recommendations_clean["weekly_summary_id"] = recommendations_clean["weekly_summary_id"].astype("Int64")
recommendations_clean["created_at"] = recommendations_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

recommendations_clean = recommendations_clean[
    [
        "id", "user_id", "stress_prediction_id", "weekly_summary_id",
        "period_type", "category", "title", "recommendation_text",
        "priority_level", "created_at"
    ]
].sort_values("id")

recommendations_clean.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,category,title,recommendation_text,priority_level,created_at
0,1,1,1,<NA>,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-02 00:03:00
1,2,1,2,<NA>,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-02 20:48:00
2,3,1,3,<NA>,daily,digital_habit,Batasi screen time,Screen time lo tinggi. Kurangi penggunaan laya...,Medium,2026-01-03 20:54:00
3,4,1,4,<NA>,daily,mood_regulation,Stabilkan mood,Mood lo sedang rendah. Coba journaling singkat...,Medium,2026-01-05 01:01:00
4,5,1,5,<NA>,daily,workload,Atur prioritas tugas,Tekanan akademik lo tinggi. Pecah tugas jadi 3...,High,2026-01-05 23:14:00


## Insight:
`recommendations_clean` tidak dibuat ulang.

Cleaning hanya memastikan:
- `period_type` valid
- source id valid
- expected missing tetap sesuai desain daily/weekly
- relasi ke `stress_predictions_clean` atau `weekly_summaries_clean` tetap aman

## 19. Cleaning `insights`

In [43]:
insights_clean = insights.copy()

# Parse tipe data
insights_clean["id"] = pd.to_numeric(insights_clean["id"], errors="coerce")
insights_clean["user_id"] = pd.to_numeric(insights_clean["user_id"], errors="coerce")
insights_clean["stress_prediction_id"] = pd.to_numeric(insights_clean["stress_prediction_id"], errors="coerce")
insights_clean["weekly_summary_id"] = pd.to_numeric(insights_clean["weekly_summary_id"], errors="coerce")
insights_clean["period_type"] = insights_clean["period_type"].astype(str).str.strip().str.lower()
insights_clean["insight_text"] = insights_clean["insight_text"].astype(str).str.strip()
insights_clean["created_at"] = pd.to_datetime(insights_clean["created_at"], errors="coerce")

# Hapus data fatal umum
insights_clean = insights_clean.dropna(
    subset=["id", "user_id", "period_type", "insight_text", "created_at"]
)

# Validasi user dan period_type
insights_clean = insights_clean[
    insights_clean["user_id"].isin(valid_user_ids)
]

insights_clean = insights_clean[
    insights_clean["period_type"].isin(["daily", "weekly"])
]

# Validasi source id sesuai period_type
daily_mask = insights_clean["period_type"] == "daily"
weekly_mask = insights_clean["period_type"] == "weekly"

daily_valid = (
    daily_mask
    & insights_clean["stress_prediction_id"].isin(valid_prediction_ids)
    & insights_clean["weekly_summary_id"].isna()
)

weekly_valid = (
    weekly_mask
    & insights_clean["weekly_summary_id"].isin(valid_weekly_ids)
    & insights_clean["stress_prediction_id"].isna()
)

insights_clean = insights_clean[daily_valid | weekly_valid]

# Format final
insights_clean["id"] = insights_clean["id"].astype(int)
insights_clean["user_id"] = insights_clean["user_id"].astype(int)

insights_clean["stress_prediction_id"] = insights_clean["stress_prediction_id"].astype("Int64")
insights_clean["weekly_summary_id"] = insights_clean["weekly_summary_id"].astype("Int64")
insights_clean["created_at"] = insights_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

insights_clean = insights_clean[
    [
        "id", "user_id", "stress_prediction_id", "weekly_summary_id",
        "period_type", "insight_text", "created_at"
    ]
].sort_values("id")

insights_clean.head()

,id,user_id,stress_prediction_id,weekly_summary_id,period_type,insight_text,created_at
0,1,1,1,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 00:01:00
1,2,1,2,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-02 20:46:00
2,3,1,3,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-03 20:52:00
3,4,1,4,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 00:59:00
4,5,1,5,<NA>,daily,Stress score hari ini sedang. Faktor yang perl...,2026-01-05 23:12:00


## Insight:
`insights_clean` tidak dibuat ulang.

Cleaning hanya memvalidasi source id dan mempertahankan expected missing sesuai desain daily/weekly.

# D. Final Validation dan Save Output

## 20. Final Validation

In [44]:
validation_rows = []

def add_validation(rule, passed, details):
    validation_rows.append({
        "rule": rule,
        "passed": bool(passed),
        "details": details
    })

add_validation(
    "users.id unique",
    users_clean["id"].is_unique,
    f"{users_clean['id'].nunique()} unique ids / {len(users_clean)} rows"
)

add_validation(
    "authentications.user_id exists in users",
    set(authentications_clean["user_id"]).issubset(set(users_clean["id"])),
    "FK users -> authentications"
)

add_validation(
    "daily_activities.id unique",
    daily_clean["id"].is_unique,
    f"{daily_clean['id'].nunique()} unique ids / {len(daily_clean)} rows"
)

add_validation(
    "daily_activities user_id + activity_date unique",
    not daily_clean.duplicated(["user_id", "activity_date"]).any(),
    f"{daily_clean.duplicated(['user_id', 'activity_date']).sum()} duplicate keys"
)

add_validation(
    "daily_activities social_media_hours <= screen_time_hours",
    (daily_clean["social_media_hours"] <= daily_clean["screen_time_hours"]).all(),
    "logical consistency check"
)

add_validation(
    "stress_predictions.activity_id exists in daily_activities",
    set(stress_predictions_clean["activity_id"]).issubset(set(daily_clean["id"])),
    "FK daily_activities -> stress_predictions"
)

add_validation(
    "stress_predictions.activity_id unique",
    stress_predictions_clean["activity_id"].is_unique,
    f"{stress_predictions_clean['activity_id'].nunique()} unique activity ids / {len(stress_predictions_clean)} rows"
)

add_validation(
    "stress_score between 0 and 100",
    stress_predictions_clean["stress_score"].between(0, 100).all(),
    f"min={stress_predictions_clean['stress_score'].min()}, max={stress_predictions_clean['stress_score'].max()}"
)

add_validation(
    "stress_level valid",
    stress_predictions_clean["stress_level"].isin(["Low", "Medium", "High"]).all(),
    str(stress_predictions_clean["stress_level"].value_counts().to_dict())
)

add_validation(
    "weekly_summaries user_id + week_start + week_end unique",
    not weekly_summaries_clean.duplicated(["user_id", "week_start", "week_end"]).any(),
    f"{weekly_summaries_clean.duplicated(['user_id', 'week_start', 'week_end']).sum()} duplicate keys"
)

add_validation(
    "recommendations period_type valid",
    recommendations_clean["period_type"].isin(["daily", "weekly"]).all(),
    str(recommendations_clean["period_type"].value_counts().to_dict())
)

add_validation(
    "insights period_type valid",
    insights_clean["period_type"].isin(["daily", "weekly"]).all(),
    str(insights_clean["period_type"].value_counts().to_dict())
)

# recommendations source validation
rec_daily = recommendations_clean["period_type"] == "daily"
rec_weekly = recommendations_clean["period_type"] == "weekly"

rec_daily_valid = (
    recommendations_clean.loc[rec_daily, "stress_prediction_id"].notna()
    & recommendations_clean.loc[rec_daily, "weekly_summary_id"].isna()
).all()

rec_weekly_valid = (
    recommendations_clean.loc[rec_weekly, "weekly_summary_id"].notna()
    & recommendations_clean.loc[rec_weekly, "stress_prediction_id"].isna()
).all()

add_validation(
    "recommendations source id valid",
    rec_daily_valid and rec_weekly_valid,
    "daily uses stress_prediction_id; weekly uses weekly_summary_id"
)

# insights source validation
ins_daily = insights_clean["period_type"] == "daily"
ins_weekly = insights_clean["period_type"] == "weekly"

ins_daily_valid = (
    insights_clean.loc[ins_daily, "stress_prediction_id"].notna()
    & insights_clean.loc[ins_daily, "weekly_summary_id"].isna()
).all()

ins_weekly_valid = (
    insights_clean.loc[ins_weekly, "weekly_summary_id"].notna()
    & insights_clean.loc[ins_weekly, "stress_prediction_id"].isna()
).all()

add_validation(
    "insights source id valid",
    ins_daily_valid and ins_weekly_valid,
    "daily uses stress_prediction_id; weekly uses weekly_summary_id"
)

validation_summary = pd.DataFrame(validation_rows)
validation_summary

,rule,passed,details
0,users.id unique,True,300 unique ids / 300 rows
1,authentications.user_id exists in users,True,FK users -> authentications
2,daily_activities.id unique,True,26984 unique ids / 26984 rows
3,daily_activities user_id + activity_date unique,True,0 duplicate keys
4,daily_activities social_media_hours <= screen_...,True,logical consistency check
5,stress_predictions.activity_id exists in daily...,True,FK daily_activities -> stress_predictions
6,stress_predictions.activity_id unique,True,26579 unique activity ids / 26579 rows
7,stress_score between 0 and 100,True,"min=21.0, max=92.83"
8,stress_level valid,True,"{'Medium': 23256, 'High': 2583, 'Low': 740}"
9,weekly_summaries user_id + week_start + week_e...,True,0 duplicate keys


## 21. Save Clean Dataset ke `data/processed/`

In [45]:
processed_files = {
    "users_clean.csv": users_clean,
    "authentications_clean.csv": authentications_clean,
    "daily_activities_clean.csv": daily_clean,
    "stress_predictions_clean.csv": stress_predictions_clean,
    "weekly_summaries_clean.csv": weekly_summaries_clean,
    "recommendations_clean.csv": recommendations_clean,
    "insights_clean.csv": insights_clean
}

for file_name, df in processed_files.items():
    df.to_csv(PROCESSED_DIR / file_name, index=False)

validation_summary.to_csv(REPORT_DIR / "validation_summary.csv", index=False)

for file_name, df in processed_files.items():
    print(file_name, df.shape)

users_clean.csv (300, 7)
authentications_clean.csv (300, 6)
daily_activities_clean.csv (26984, 18)
stress_predictions_clean.csv (26579, 7)
weekly_summaries_clean.csv (3600, 13)
recommendations_clean.csv (26784, 10)
insights_clean.csv (29551, 7)


## 22. Buat Wrangling Report

In [46]:
def dataframe_to_markdown(df):
    # Membuat tabel markdown sederhana tanpa package tabulate
    df = df.copy().astype(str)

    header = "| " + " | ".join(df.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df.columns)) + " |"

    rows = []
    for _, row in df.iterrows():
        rows.append("| " + " | ".join(row.values) + " |")

    return "\n".join([header, separator] + rows)


report = "# Data Wrangling Report - Student Stress Detector\n\n"

report += "## 1. Raw Dataset Overview\n\n"
report += dataframe_to_markdown(overview_df)
report += "\n\n"

report += "## 2. Problem Summary\n\n"
report += dataframe_to_markdown(problem_summary)
report += "\n\n"

report += "## 3. Cleaning Summary\n\n"
report += "- users: trim whitespace, standardize email, remove duplicate id/email.\n"
report += "- authentications: validate user_id, token, expires_at, and timestamps.\n"
report += "- daily_activities: parse date, parse numeric values, impute missing values, fix range, fix social_media_hours, and deduplicate user_id + activity_date.\n"
report += "- stress_predictions: cleaned and validated without recalculating stress_score.\n"
report += "- weekly_summaries: cleaned and validated without recalculating weekly aggregation.\n"
report += "- recommendations and insights: cleaned and validated using expected daily/weekly source id rules.\n\n"

report += "## 4. Row Count Result\n\n"
row_count = pd.DataFrame([
    {"dataset": name, "rows": len(df)}
    for name, df in processed_files.items()
])
report += dataframe_to_markdown(row_count)
report += "\n\n"

report += "## 5. Validation Summary\n\n"
report += dataframe_to_markdown(validation_summary)
report += "\n\n"

report += "## 6. Important Note\n\n"
report += "Notebook ini tidak menghitung ulang stress_score dan tidak membuat ulang stress_level.\n"
report += "modelling_dataset.csv belum dibuat di notebook ini. File tersebut dibuat pada notebook 03_modelling_dataset_preparation.ipynb.\n"

(REPORT_DIR / "data_wrangling_report.md").write_text(report, encoding="utf-8")

print("Report saved to:", REPORT_DIR / "data_wrangling_report.md")

Report saved to: C:\Users\User\Downloads\student_stress_data_science_pure_wrangling_project\student_stress_data_science_pure_wrangling\outputs\reports\data_wrangling_report.md


## 23. Final Output Check

In [47]:
print("Processed files:")
for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print("-", file.name)

print("\nReport files:")
for file in sorted(REPORT_DIR.glob("*")):
    print("-", file.name)

Processed files:
- authentications_clean.csv
- daily_activities_clean.csv
- insights_clean.csv
- recommendations_clean.csv
- stress_predictions_clean.csv
- users_clean.csv
- weekly_summaries_clean.csv

Report files:
- data_wrangling_report.md
- validation_summary.csv
